In [1]:
import importlib
import sys
import numpy as np
from collections import Counter
import pandas as pd

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load decision-labeled data
file_path_train = '../../../../../../data/Helpdesk/tensor_data/decision_labeled/helpdesk_all_5_train.pkl'
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/Helpdesk/tensor_data/decision_labeled/helpdesk_all_5_val.pkl'
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# Helpdesk dataset dynamic categories, features:
helpdesk_all_categories = helpdesk_train_dataset.all_categories

helpdesk_all_categories_cat = helpdesk_all_categories[0]
# print(helpdesk_all_categories_cat)
helpdesk_all_categories_num = helpdesk_all_categories[1]
# print(helpdesk_all_categories_num)
for i, cat in enumerate(helpdesk_all_categories_cat):
     print(f"Helpdesk dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(helpdesk_all_categories_num):
     print(f"Helpdesk dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of numerical: {num[1]}")
print('\n')
     
# Helpdesk dataset static categories, features:
helpdesk_all_stat_categories = helpdesk_train_dataset.all_static_categories

helpdesk_all_stat_categories_cat = helpdesk_all_stat_categories[0]
# print(helpdesk_all_stat_categories_cat)
helpdesk_all_stat_categories_num = helpdesk_all_stat_categories[1]
# print(helpdesk_all_stat_categories_num)
for i, cat in enumerate(helpdesk_all_stat_categories_cat):
     print(f"Helpdesk static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(helpdesk_all_stat_categories_num):
     print(f"Helpdesk static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Helpdesk amount of numerical: {num[1]}")  

Helpdesk dynamic categorical feature: Activity, Index position in categorical data list: 0
Helpdesk amount of category labels: 15
Helpdesk dynamic categorical feature: Resource, Index position in categorical data list: 1
Helpdesk amount of category labels: 24


Helpdesk dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
Helpdesk amount of numerical: 1
Helpdesk dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
Helpdesk amount of numerical: 1
Helpdesk dynamic numerical feature: day_in_week, Index position in categorical data list: 2
Helpdesk amount of numerical: 1
Helpdesk dynamic numerical feature: seconds_in_day, Index position in categorical data list: 3
Helpdesk amount of numerical: 1


Helpdesk static categorical feature: VariantIndex, Index position in categorical data list: 0
Helpdesk amount of category labels: 175
Helpdesk static categorical feature: seriousness, Index position in categorical data list:

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['Activity', 'Resource']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (unused by C-LSTM training cell, kept for consistency)
dec_feat_cat = ['Activity']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

# Guard tensors are pre-computed and stored in the dataset .pkl files
# (prepared during data loading via prepare_guard_tensors).
print(f"Guard tensors: targets {helpdesk_train_dataset._guard_targets.shape}, "
      f"mask {helpdesk_train_dataset._guard_mask.shape}, "
      # is that still present?
      f"confidence {helpdesk_train_dataset._guard_confidence.shape}")

Input features encoder:  [['Activity', 'Resource'], ['case_elapsed_time']]
Features decoder:  [['Activity'], []]
Guard tensors: targets torch.Size([13888, 18, 15]), mask torch.Size([13888, 18]), confidence torch.Size([13888, 18])


In [5]:
import suffix_pred.models.C_LSTM
importlib.reload(suffix_pred.models.C_LSTM)
from suffix_pred.models.C_LSTM import FullShared_Join_LSTM

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# C-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'Activity'
activity_feature_index = next(i for i, cat in enumerate(helpdesk_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = helpdesk_all_categories_cat[activity_feature_index][1]

# C-LSTM model initialization
model = FullShared_Join_LSTM(
    data_set_categories=helpdesk_all_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    model_feat=model_feat,
    input_size=input_size,
    output_size_act=output_size_act,
)

Data set categories:  ([('Activity', 15, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'EOS': 4, 'INVALID': 5, 'Insert ticket': 6, 'RESOLVED': 7, 'Require upgrade': 8, 'Resolve SW anomaly': 9, 'Resolve ticket': 10, 'Schedule intervention': 11, 'Take in charge ticket': 12, 'VERIFIED': 13, 'Wait': 14}), ('Resource', 24, {'EOS': 1, 'Value 1': 2, 'Value 10': 3, 'Value 11': 4, 'Value 12': 5, 'Value 13': 6, 'Value 14': 7, 'Value 15': 8, 'Value 16': 9, 'Value 17': 10, 'Value 18': 11, 'Value 19': 12, 'Value 2': 13, 'Value 20': 14, 'Value 21': 15, 'Value 22': 16, 'Value 3': 17, 'Value 4': 18, 'Value 5': 19, 'Value 6': 20, 'Value 7': 21, 'Value 8': 22, 'Value 9': 23})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {})])
Model input features:  [['Activity', 'Resource'], ['case_elapsed_time']]


Embeddings:  ModuleList(
  (0): Embedding(15, 7)
  (1): Embedding(24, 9)
)
Total embedding feature size:  16
Input feature si

/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import CTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 1e-5
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.AdamW(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)

# Epochs / Batch size
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

optimize_values = {
    "optimizer": optimizer,
    "scheduler": scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
}

# Activity feature index and EOS id for activity-only next-event prediction
activity_feature_name = 'Activity'
concept_name_id = next(i for i, cat in enumerate(helpdesk_all_categories_cat) if cat[0] == activity_feature_name)
activity_label_to_id = helpdesk_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')
if eos_id is None:
    raise ValueError("Could not find EOS id in activity label mapping.")

# Decision-aware guard regularization weight (λ_g)
lambda_g = 1.0
print(f"Decision-aware training: λ_g = {lambda_g}")

model_output_path = "../../../../../../models/Helpdesk/decision/Helpdesk_C_LSTM_v1_DA.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = CTraining(
    device=device,
    model=model,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    optimize_values=optimize_values,
    concept_name_id=concept_name_id,
    eos_id=eos_id,
    loss_obj=loss_obj,
    lambda_g=lambda_g,
    save_model_n_th_epoch=1,
    saving_path=model_output_path,
)

# Train the model (decision-aware: L_base + λ_g * L_guard)
trainer.train()

Device: cuda
Decision-aware training: λ_g = 1.0
Device:  cuda
Optimizer:  AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 1e-05
    maximize: False
    weight_decay: 0.0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7fcca9919c10>
Epochs:  100
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 1e-05
Training: Avg Attenuated Training Loss: 5.4568
Validation: Avg Validation Loss: 2.7152
Validation Loss for Scheduler: 2.7152
saving model
Epoch [2/100], Learning Rate: 1e-05
Training: Avg Attenuated Training Loss: 5.2164
Validation: Avg Validation Loss: 2.6324
Validation Loss for Scheduler: 2.6324
saving model
Epoch [3/100], Learning Rate: 1e-05
Training: Avg Attenuated Training Loss: 5.0637
Validation: Avg Validation Loss: 2.5514
Validation Loss for Scheduler: 2.5514
saving model
Epoch [4/100], Learning Rate: 1e-05
Training: Avg Attenuated Training Loss: 4.7946
Validation: Avg Validation Loss: 2.4792
Validation Loss for Scheduler: 2.4792
saving model
Epoch [5/100], Learning Rate: 1e-05
Training: Avg Attenuated Training Loss: 4.7414
Validation: Avg Validation Loss: 2.3830
Validation Loss for Scheduler: 2.3830
saving model
Epoch [6/100], Learning Rate: 1e-05
Training: Avg Attenuated Training Loss: 4.4907
Validation: Avg Validation Loss: 2.2119
Validat

([5.456787947121017,
  5.216379067219726,
  5.063738617328329,
  4.794557282684046,
  4.741386907909988,
  4.490674760363517,
  4.2907703841498135,
  4.063076780476702,
  3.850158506577168,
  3.5949491503041817,
  3.3576355691349833,
  3.137558006365365,
  3.0157227789590118,
  2.8134243138339543,
  2.536873369041933,
  2.4098402600769604,
  2.2259355203821025,
  2.1167312608946354,
  1.9901149076059323,
  1.9308839686419985,
  1.8319519840249228,
  1.7186819479006146,
  1.7108915316949196,
  1.6943826877742731,
  1.6131412452514018,
  1.6837081067059019,
  1.4996995521247933,
  1.4698521150361508,
  1.4041982615759614,
  1.4239868527158686,
  1.3975873687945375,
  1.357901861908239,
  1.3183598791787383,
  1.3709076309422834,
  1.3929699391400048,
  1.352364359098837,
  1.2993745303482092,
  1.2023704866750524,
  1.15081197460857,
  1.1576279032667842,
  1.2035749289420767,
  1.1933897099363695,
  1.249252194658332,
  1.180339747065798,
  1.09682933398343,
  1.0663573504041095,
  1.07